# Local July ST Circulant Simulation Generator — smooth=0.3

Moved-location local runner for:

```text
/Users/joonwonlee/Documents/GEMS_TCO-1/simulate_data/generate_july_st_circulant_real_locations_2022_2025_smooth0p3_051926.py
```

This notebook writes generated simulation assets outside the git repo under:

```text
/Users/joonwonlee/Documents/GEMS_DATA/simulation/july_st_circulant_realpattern_smooth0p3
```

It is intentionally **not auto-running** because the full July 2022-2025 generation is slow.
Set `RUN_NOW = True` only after checking paths and output directory.

The runner prepends the current repository's `src` directory and prefers the local `gems_tco` conda interpreter. Set `GEMS_TCO_PYTHON` to override that interpreter.

Smooth-specific behavior:
- `smooth=0.5`: closed-form exponential Matérn correlation inside the generator
- `smooth=0.3` or `smooth=1.0`: the shared `GEMS_TCO.spatial.matern_spline` implementation


In [ ]:
import os
import sys
import shlex
import subprocess
from pathlib import Path

def find_repo_root():
    starts = [Path.cwd(), Path('/Users/joonwonlee/Documents/GEMS_TCO-1')]
    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'GEMS_TCO').is_dir():
                return candidate
    raise FileNotFoundError('Could not locate the GEMS_TCO repository root.')


def choose_python():
    configured = os.environ.get('GEMS_TCO_PYTHON')
    candidates = ([Path(configured)] if configured else []) + [
        Path('/opt/anaconda3/envs/gems_tco/bin/python'),
        Path(sys.executable),
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError('No Python interpreter found. Set GEMS_TCO_PYTHON.')


ROOT = find_repo_root()
PYTHON = choose_python()
GENERATOR = ROOT / 'simulate_data' / 'generate_july_st_circulant_real_locations_2022_2025_smooth0p3_051926.py'
INPUT_ROOT = Path('/Users/joonwonlee/Documents/GEMS_DATA')
OUTPUT_DIR = Path('/Users/joonwonlee/Documents/GEMS_DATA') / 'simulation' / 'july_st_circulant_realpattern_smooth0p3'

SMOOTH = 0.3
YEARS = (2022, 2023, 2024, 2025)
MAX_HOURS = 248
HOURS_PER_DAY = 8
SEED = 20240701

# Full run is slow. For a quick local smoke test, set YEARS=(2024,) and MAX_HOURS=8.
RUN_NOW = False

assert GENERATOR.exists(), f'Missing generator: {GENERATOR}'
print('Python:', PYTHON)
print('Repository:', ROOT)
print('Generator:', GENERATOR)
print('Input root:', INPUT_ROOT, 'exists=', INPUT_ROOT.exists())
print('Output dir:', OUTPUT_DIR)
print('Smooth:', SMOOTH)


In [ ]:
expected_inputs = [
    INPUT_ROOT / f'pickle_{year}' / f'tco_grid_{str(year)[2:]}_07.pkl'
    for year in YEARS
]
for p in expected_inputs:
    print(('OK   ' if p.exists() else 'MISS '), p)
missing_inputs = [p for p in expected_inputs if not p.exists()]
if missing_inputs:
    raise FileNotFoundError(f'Missing {len(missing_inputs)} input pickle(s).')


## Build command

The command writes one output folder per year and creates both real-location and gridded pickles.


In [ ]:
env = os.environ.copy()
pythonpath = [str(ROOT / 'src')]
if env.get('PYTHONPATH'):
    pythonpath.append(env['PYTHONPATH'])
env['PYTHONPATH'] = os.pathsep.join(pythonpath)
for name in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    env.setdefault(name, '4')

check = subprocess.run(
    [str(PYTHON), '-c', "import GEMS_TCO, torch; print('GEMS_TCO:', GEMS_TCO.__file__); print('torch:', torch.__version__)"],
    cwd=ROOT, env=env, check=True, capture_output=True, text=True,
)
print(check.stdout.strip())

cmd = [
    str(PYTHON), str(GENERATOR),
    '--years', ','.join(map(str, YEARS)),
    '--input-root', str(INPUT_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--max-hours', str(MAX_HOURS),
    '--hours-per-day', str(HOURS_PER_DAY),
    '--seed', str(SEED),
    '--smooth', str(SMOOTH),
    '--spline-n-points', '4000',
    '--spline-r-max', '30.0',
    '--sigmasq', '10.0',
    '--range-lat', '0.2',
    '--range-lon', '0.3',
    '--range-time', '2.0',
    '--advec-lat', '0.08',
    '--advec-lon', '-0.2',
    '--nugget', '1.0',
    '--mean-intercept', '260.0',
    '--mean-lat-slope', '1.0',
    '--mean-lat-center', '-0.5',
    '--lat-range=-3,2',
    '--lon-range=121,131',
    '--lat-factor-hr', '100',
    '--lon-factor-hr', '10',
    '--hr-pad', '0.1',
]
print(' '.join(shlex.quote(x) for x in cmd))


In [ ]:
if RUN_NOW:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(cmd, cwd=ROOT, env=env, check=True)
else:
    print('RUN_NOW is False. Set RUN_NOW=True to generate data.')


## Verify outputs


In [ ]:
if OUTPUT_DIR.exists():
    for year_dir in sorted(OUTPUT_DIR.glob('*_july_st_circulant')):
        files = sorted(p.name for p in year_dir.iterdir())
        print()
        print(year_dir)
        for name in files:
            print('  ', name)
else:
    print('No output directory yet:', OUTPUT_DIR)
